

# Statistical Analyses:


1. Survival analysis: Time-to-event modeling for NT→NZ conversion. Cox proportional hazards to identify what predicts faster conversion.

2. Markov chains: Model state transitions (No commitment → NT:C → NT:T → NT+NZ). Calculate transition probabilities, steady-state distributions, expected time in each state.

3. Logistic regression: Predict which companies will convert NT→NZ based on cohort year, sector, region, initial status type.

4. Clustering: Group companies by their trajectory patterns (fast adopters, slow movers, dropouts, leapfroggers).

5. Churn analysis: Model why companies lose commitments (those 109 NZ losses in 2024→2025).

#### Temporal Analyses

6. Acceleration metrics: Is adoption speeding up? Compare slopes between cohorts.

7. Momentum indicators: Leading vs lagging sectors/regions in adoption waves.

8. Seasonality: Do commitments cluster around specific times (COP meetings, reporting cycles)?

#### Network/Portfolio Analyses

9. Portfolio risk: If X% typically drop targets, what's the expected stable state?


10. Contagion effects: If you had company relationships, model peer influence on adoption.

11. Optimal pathway: Which progression sequence has highest retention? (Direct to both vs stepwise)

#### Predictive Models

12. Time series forecasting: Project 2026-2030 adoption rates using ARIMA or exponential smoothing.

13. Cohort retention curves: Kaplan-Meier style plots showing retention by entry year.

14. Propensity scoring: Given attributes, probability of NT→NZ within 1/2/3 years.

#### What Would Be Most Insightful?

Given data quality, I'd prioritize:
- Markov chain model (clean state transitions, interpretable probabilities)
- Survival analysis (directly answers "when will they convert")
- Churn analysis (explains the anomalous NZ losses)





## What correlations to test:

1. Carbon credit usage vs commitment types
   - Companies saying they'll use carbon credits → higher likelihood of having CN/NZ/SBT?
   - Does CC usage correlate with faster NT→NZ conversion?

2. Commitment co-occurrence
   - If you have SBT, how likely to also have NZ?
   - If you have CN, how likely to mention CC usage?
   - Which commitments cluster together?

3. Temporal patterns
   - Does CC usage percentage change over time?
   - Does CC acceptance correlate with cohort year?

4. Regional/sectoral patterns
   - Which regions/sectors more likely to use CCs?
   - Does this correlate with commitment types?

## Statistical tests:

- Chi-square test: Independence between categorical variables (CC yes/no × SBT yes/no)
- Cramér's V: Strength of association (0-1 scale)
- Phi coefficient: For 2×2 tables specifically
- Point-biserial correlation: Binary (CC yes/no) vs continuous (number of commitments)
- Tetrachoric correlation: Underlying continuous relationship between two binary variables

## Example output:

"Companies using carbon credits are 2.3x more likely to have NZ targets (χ²=45.3, p<0.001, Cramér's V=0.28)"


## Markov 


In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

df = pd.read_excel('historic_new.xlsx', sheet_name='sbti evolution ')
df.columns = df.columns.str.strip()
df['company'] = df['company'].astype(str)

years = ['2021', '2022', '2023', '2024', '2025']

# Define states
def get_state(nt, nz):
    nt = str(nt) if pd.notna(nt) else 'None'
    nz = str(nz) if pd.notna(nz) else 'None'
    
    if nt == 'None' and nz == 'None':
        return 'No commitment'
    if nt != 'None' and nz == 'None':
        return f'NT:{nt}'
    if nt == 'None' and nz != 'None':
        return f'NZ:{nz}'
    return f'NT:{nt}+NZ:{nz}'

# Build states for each company-year
states = {}
for _, row in df.iterrows():
    company = row['company']
    states[company] = {}
    for year in years:
        nt = row[f'{year}_NT_Status']
        nz = row[f'{year}_NZ_Status']
        states[company][year] = get_state(nt, nz)

# Count transitions
transitions = {}
for company in states:
    for i in range(len(years) - 1):
        from_state = states[company][years[i]]
        to_state = states[company][years[i+1]]
        
        if from_state not in transitions:
            transitions[from_state] = {}
        if to_state not in transitions[from_state]:
            transitions[from_state][to_state] = 0
        
        transitions[from_state][to_state] += 1

# Get all unique states
all_states = sorted(set(s for company in states.values() for s in company.values()))

# Build transition matrix
n = len(all_states)
matrix = np.zeros((n, n))
state_to_idx = {s: i for i, s in enumerate(all_states)}

for from_state in transitions:
    from_idx = state_to_idx[from_state]
    row_sum = sum(transitions[from_state].values())
    
    for to_state, count in transitions[from_state].items():
        to_idx = state_to_idx[to_state]
        matrix[from_idx, to_idx] = count / row_sum

# Print transition matrix
print("TRANSITION PROBABILITY MATRIX")
print("Rows = current state, Columns = next state\n")

# Print header
print(f"{'From State':<20}", end="")
for state in all_states:
    print(f"{state:<20}", end="")
print()

# Print matrix
for i, from_state in enumerate(all_states):
    print(f"{from_state:<20}", end="")
    for j in range(n):
        if matrix[i, j] > 0:
            print(f"{matrix[i, j]:.3f}              ", end="")
        else:
            print(f"{'.':<20}", end="")
    print()

# Steady state (eigenvector for eigenvalue 1)
eigenvalues, eigenvectors = np.linalg.eig(matrix.T)
steady_idx = np.argmax(np.abs(eigenvalues - 1.0) < 1e-10)
steady = np.real(eigenvectors[:, steady_idx])
steady = steady / steady.sum()

print("\n\nSTEADY STATE DISTRIBUTION")
print("Long-run equilibrium probabilities:\n")
for i, state in enumerate(all_states):
    print(f"{state:<30} {steady[i]:.3f} ({steady[i]*100:.1f}%)")

# Key transitions
print("\n\nKEY TRANSITIONS (>5% probability)")
key = []
for i, from_state in enumerate(all_states):
    for j, to_state in enumerate(all_states):
        if matrix[i, j] > 0.05 and i != j:
            key.append((from_state, to_state, matrix[i, j]))

key.sort(key=lambda x: -x[2])
for from_s, to_s, prob in key:
    print(f"{from_s:<25} → {to_s:<25} {prob:.3f}")

# Visualization: Sankey of top transitions
sources = []
targets = []
values = []
labels = all_states.copy()

for i, from_state in enumerate(all_states):
    for j, to_state in enumerate(all_states):
        if matrix[i, j] > 0.02:
            sources.append(i)
            targets.append(j)
            values.append(matrix[i, j])

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        label=labels,
        color='lightblue'
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values
    )
)])

fig.update_layout(
    title="Markov Chain Transition Probabilities (edges > 2%)",
    height=800,
    width=1200
)

import os
os.chdir('/mnt/user-data/outputs')
fig.write_html('markov_transitions.html')

# Save transition matrix
tm_df = pd.DataFrame(matrix, index=all_states, columns=all_states)
tm_df.to_csv('transition_matrix.csv')

# Save steady state
ss_df = pd.DataFrame({'state': all_states, 'probability': steady})
ss_df.to_csv('steady_state.csv', index=False)

print("\n\nFiles: markov_transitions.html, transition_matrix.csv, steady_state.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'historic_new.xlsx'

## Cross-sectional analyses (2025 only):

Chi-square independence tests: CC usage vs NZ, CC vs SBT, etc.
Cramér's V correlation matrix: Heatmap of all commitment associations
Conditional probabilities: P(NZ | SBT), P(CC | CN), etc.
Logistic regression: Predict NZ from sector, region, CC, NT status
Cluster analysis: Group companies by commitment profile
Sector/region benchmarking: Which sectors lead in each commitment type

## Longitudinal with 2024-2025:

Year-over-year changes: Who gained/lost each commitment
Transition analysis: 2024 state → 2025 state (9x9 matrix)
Upgrade/downgrade rates: C→T vs T→C vs dropouts
Logistic for change: Predict who converts/drops based on 2024 profile

# 2025 only stats


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
import plotly.graph_objects as go

df = pd.read_excel(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\historic new .xlsx', sheet_name='2025', header=1, nrows=500)
df.columns = ['company', 're100', 'has_nt', 'gov_cn', 'cn', 'nz', 'cc_yes', 'cc_no', 'any_action']
df = df.drop(0).reset_index(drop=True)

for col in ['re100', 'has_nt', 'gov_cn', 'cn', 'nz', 'cc_yes', 'cc_no', 'any_action']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

vars_test = ['has_nt', 'nz', 'cc_yes', 'cn', 're100']

print("CHI-SQUARE")
for i, v1 in enumerate(vars_test):
    for v2 in vars_test[i+1:]:
        ct = pd.crosstab(df[v1], df[v2])
        chi2, p, _, _ = chi2_contingency(ct)
        v = np.sqrt(chi2 / (ct.sum().sum() * (min(ct.shape) - 1)))
        print(f"{v1} vs {v2}: V={v:.3f}, p={p:.4f}")

print("\nCONDITIONAL PROB")
print(f"P(nz|has_nt) = {df.loc[df['has_nt']==1, 'nz'].mean():.3f}")
print(f"P(nz|cc_yes) = {df.loc[df['cc_yes']==1, 'nz'].mean():.3f}")
print(f"P(cc_yes|cn) = {df.loc[df['cn']==1, 'cc_yes'].mean():.3f}")
print(f"P(has_nt|re100) = {df.loc[df['re100']==1, 'has_nt'].mean():.3f}")

print("\nLOGISTIC: PREDICT NZ")
X = df[['has_nt', 'cc_yes', 'cn', 're100', 'nz']]
lr = LogisticRegression(max_iter=1000)
lr.fit(X, df['nz'])
for feat, coef in zip(X.columns, lr.coef_[0]):
    print(f"{feat}: OR={np.exp(coef):.2f}")
print("\nLOGISTIC: PREDICT CC")
lr.fit(X, df['cc_yes'])
for feat, coef in zip(X.columns, lr.coef_[0]):
    print(f"{feat}: OR={np.exp(coef):.2f}")

print("\nCLUSTERS")
km = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(df[vars_test])
for i in range(4):
    n = (df['cluster']==i).sum()
    means = df[df['cluster']==i][vars_test].mean()
    print(f"C{i} (n={n}): nt={means['has_nt']:.2f}, nz={means['nz']:.2f}, cc={means['cc_yes']:.2f}, cn={means['cn']:.2f}, re={means['re100']:.2f}")

n = len(vars_test)
corr = np.zeros((n, n))
for i, v1 in enumerate(vars_test):
    for j, v2 in enumerate(vars_test):
        if i == j:
            corr[i,j] = 1.0
        else:
            ct = pd.crosstab(df[v1], df[v2])
            chi2, _, _, _ = chi2_contingency(ct)
            corr[i,j] = np.sqrt(chi2 / (ct.sum().sum() * (min(ct.shape) - 1)))

print("\nCRAMERS V")
print(pd.DataFrame(corr, index=vars_test, columns=vars_test).round(3))

fig = go.Figure(go.Heatmap(z=corr, x=vars_test, y=vars_test, colorscale='Blues', text=np.round(corr, 3), texttemplate='%{text}'))
fig.update_layout(title="Cramers V", height=600, width=700)
fig.write_html('cramers_v.html')




CHI-SQUARE
has_nt vs nz: V=0.309, p=0.0000
has_nt vs cc_yes: V=0.099, p=0.0265
has_nt vs cn: V=0.044, p=0.3234
has_nt vs re100: V=0.280, p=0.0000
nz vs cc_yes: V=0.435, p=0.0000
nz vs cn: V=0.472, p=0.0000
nz vs re100: V=0.284, p=0.0000
cc_yes vs cn: V=0.040, p=0.3748
cc_yes vs re100: V=0.150, p=0.0008
cn vs re100: V=0.065, p=0.1449

CONDITIONAL PROB
P(nz|has_nt) = 0.737
P(nz|cc_yes) = 0.751
P(cc_yes|cn) = 0.396
P(has_nt|re100) = 0.623

LOGISTIC: PREDICT NZ
has_nt: OR=1.79
cc_yes: OR=2.44
cn: OR=0.26
re100: OR=1.66
nz: OR=628.93

LOGISTIC: PREDICT CC
has_nt: OR=0.98
cc_yes: OR=954.85
cn: OR=1.37
re100: OR=1.13
nz: OR=2.35

CLUSTERS
C0 (n=88): nt=0.60, nz=0.98, cc=0.00, cn=0.00, re=0.28
C1 (n=171): nt=0.40, nz=0.97, cc=1.00, cn=0.00, re=0.25
C2 (n=91): nt=0.26, nz=0.00, cc=0.40, cn=1.00, re=0.10
C3 (n=149): nt=0.07, nz=0.00, cc=0.09, cn=0.00, re=0.00

CRAMERS V
        has_nt     nz  cc_yes     cn  re100
has_nt   1.000  0.309   0.099  0.044  0.280
nz       0.309  1.000   0.435  0.472  0

In [4]:
print(pd.DataFrame(corr, index=vars_test, columns=vars_test).round(3))

        has_nt     nz  cc_yes     cn  re100
has_nt   1.000  0.309   0.099  0.044  0.280
nz       0.309  1.000   0.435  0.472  0.284
cc_yes   0.099  0.435   1.000  0.040  0.150
cn       0.044  0.472   0.040  1.000  0.065
re100    0.280  0.284   0.150  0.065  1.000


## overlap matrix

In [5]:
# List of key variables
vars_summary = ['nz', 'cn', 're100', 'has_nt', 'cc_yes']

# Initialize empty DataFrame for the overlap counts
overlap_matrix = pd.DataFrame(index=vars_summary, columns=vars_summary)

# Fill the matrix
for row_var in vars_summary:
    for col_var in vars_summary:
        overlap_matrix.loc[row_var, col_var] = ((df[row_var]==1) & (df[col_var]==1)).sum()

# Convert to integer
overlap_matrix = overlap_matrix.astype(int)

print(overlap_matrix)


         nz  cn  re100  has_nt  cc_yes
nz      252   0     65     115     166
cn        0  91      9      24      36
re100    65   9     77      48      48
has_nt  115  24     48     156      81
cc_yes  166  36     48      81     221


## Further Analysis 2025-2024

In [ ]:
print("ANALYSIS 1: CHI-SQUARE INDEPENDENCE TESTS (2025)")


vars_2025 = ['has_nt_2025', 'has_nz_2025', 'cc_usage', 'cn', 're100']
chi_results = []

for i, var1 in enumerate(vars_2025):
    for var2 in vars_2025[i+1:]:
        ct = pd.crosstab(df[var1], df[var2])
        chi2, p, dof, exp = chi2_contingency(ct)
        n = ct.sum().sum()
        cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
        chi_results.append({
            'var1': var1,
            'var2': var2,
            'chi2': chi2,
            'p': p,
            'cramers_v': cramers_v
        })
        print(f"{var1} vs {var2}: chi2={chi2:.2f}, p={p:.4f}, V={cramers_v:.3f}")

chi_df = pd.DataFrame(chi_results)

In [ ]:

print("\nANALYSIS 2: CONDITIONAL PROBABILITIES (2025)")


conditions = [
    ('has_nz_2025', 'has_nt_2025'),
    ('has_nz_2025', 'cc_usage'),
    ('cc_usage', 'cn'),
    ('has_nt_2025', 're100')
]

for outcome, given in conditions:
    prob = df[df[given]==1][outcome].mean()
    print(f"P({outcome} | {given}=1) = {prob:.3f}")

In [ ]:


print("\nANALYSIS 3: YEAR-OVER-YEAR CHANGES")


for var in ['has_nt', 'has_nz']:
    gained = ((df[f'{var}_2024']==0) & (df[f'{var}_2025']==1)).sum()
    lost = ((df[f'{var}_2024']==1) & (df[f'{var}_2025']==0)).sum()
    kept = ((df[f'{var}_2024']==1) & (df[f'{var}_2025']==1)).sum()
    print(f"{var}: +{gained} gained, -{lost} lost, {kept} kept")


In [ ]:

print("\nANALYSIS 4: STATE TRANSITIONS 2024->2025")


def state(row, year):
    nt = row[f'{year}_NT_Status']
    nz = row[f'{year}_NZ_Status']
    if pd.isna(nt) and pd.isna(nz): return 'None'
    if pd.notna(nt) and pd.isna(nz): return 'NT'
    if pd.isna(nt) and pd.notna(nz): return 'NZ'
    return 'Both'

df['state_2024'] = df.apply(lambda r: state(r, '2024'), axis=1)
df['state_2025'] = df.apply(lambda r: state(r, '2025'), axis=1)

trans_ct = pd.crosstab(df['state_2024'], df['state_2025'])
print(trans_ct)



## Logistic regression 2025

In [9]:
print("\nANALYSIS 5: LOGISTIC REGRESSION - PREDICT NZ_2025")


X = df[['has_nt', 'cc_yes', 'cn', 're100']].fillna(0)
y = df['nz']

lr = LogisticRegression(max_iter=1000)
lr.fit(X, y)

for feat, coef in zip(X.columns, lr.coef_[0]):
    odds_ratio = np.exp(coef)
    print(f"{feat}: coef={coef:.3f}, OR={odds_ratio:.3f}")





ANALYSIS 5: LOGISTIC REGRESSION - PREDICT NZ_2025
has_nt: coef=1.550, OR=4.709
cc_yes: coef=2.310, OR=10.071
cn: coef=-4.504, OR=0.011
re100: coef=1.351, OR=3.861


In [10]:
print("\nANALYSIS 5: LOGISTIC REGRESSION - PREDICT CC_2025")


X = df[['has_nt', 'nz', 'cn', 're100']].fillna(0)
y = df['cc_yes']

lr = LogisticRegression(max_iter=1000)
lr.fit(X, y)

for feat, coef in zip(X.columns, lr.coef_[0]):
    odds_ratio = np.exp(coef)
    print(f"{feat}: coef={coef:.3f}, OR={odds_ratio:.3f}")



ANALYSIS 5: LOGISTIC REGRESSION - PREDICT CC_2025
has_nt: coef=-0.284, OR=0.753
nz: coef=2.438, OR=11.445
cn: coef=1.311, OR=3.710
re100: coef=0.204, OR=1.227


In [11]:
import pandas as pd

# Check the actual counts
print("CROSSTAB: NZ vs CC_YES")
ct = pd.crosstab(df['nz'], df['cc_yes'], margins=True)
print(ct)

print("\n\nMANUAL ODDS CALCULATION:")

# Companies WITH NZ
nz_yes = df[df['nz']==1]
nz_has_cc = (nz_yes['cc_yes']==1).sum()
nz_no_cc = (nz_yes['cc_yes']==0).sum()
odds_nz = nz_has_cc / nz_no_cc if nz_no_cc > 0 else float('inf')
print(f"NZ companies: {nz_has_cc} have CC, {nz_no_cc} don't have CC")
print(f"Odds of CC given NZ: {odds_nz:.3f}")

# Companies WITHOUT NZ
nz_no = df[df['nz']==0]
no_nz_has_cc = (nz_no['cc_yes']==1).sum()
no_nz_no_cc = (nz_no['cc_yes']==0).sum()
odds_no_nz = no_nz_has_cc / no_nz_no_cc if no_nz_no_cc > 0 else 0
print(f"\nNo NZ companies: {no_nz_has_cc} have CC, {no_nz_no_cc} don't have CC")
print(f"Odds of CC given no NZ: {odds_no_nz:.3f}")

# Odds Ratio
if odds_no_nz > 0:
    or_manual = odds_nz / odds_no_nz
    print(f"\n**Odds Ratio (manual): {or_manual:.3f}**")
    print(f"Companies with NZ are {or_manual:.1f}x more likely to use CC")
else:
    print("\nCannot calculate OR (division by zero)")

# Compare to logistic regression result
print(f"\nLogistic Regression OR: 11.445")

CROSSTAB: NZ vs CC_YES
cc_yes    0    1  All
nz                   
0       192   55  247
1        86  166  252
All     278  221  499


MANUAL ODDS CALCULATION:
NZ companies: 166 have CC, 86 don't have CC
Odds of CC given NZ: 1.930

No NZ companies: 55 have CC, 192 don't have CC
Odds of CC given no NZ: 0.286

**Odds Ratio (manual): 6.738**
Companies with NZ are 6.7x more likely to use CC

Logistic Regression OR: 11.445


 "When you compare two companies with identical NT, CN, and RE100 status, the one with NZ is 11.4x more likely to use credits"

The simple odds ratio (6.7x) mixes in confounding effects - some of that NZ→CC association might be because NZ companies also have NT, RE100, etc.

Conservative claim: "Companies with net zero are 7x more likely to use carbon credits" (manual OR)
Controlled claim: "Net zero independently predicts 11x higher credit usage, even accounting for other climate commitments" (logistic OR)

The 6.7x is easier to verify and more intuitive. The 11.4x is technically more rigorous but harder to explain.

In [12]:
print("\nANALYSIS 6: SECTOR/REGION BENCHMARKING (2025)")


for group in ['sector', 'region']:
    print(f"\n{group}:")
    agg = df.groupby(group)[['has_nt', 'nz', 'cc_yes']].mean()
    print(agg.round(3))



ANALYSIS 6: SECTOR/REGION BENCHMARKING (2025)

sector:


KeyError: 'sector'

In [ ]:
print("\nANALYSIS 7: CLUSTER ANALYSIS (2025)")


X_cluster = df[['has_nt_2025', 'has_nz_2025', 'cc_usage', 'cn', 're100']].fillna(0)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_cluster)

for i in range(4):
    cluster_df = df[df['cluster']==i]
    n = len(cluster_df)
    profile = cluster_df[['has_nt_2025', 'has_nz_2025', 'cc_usage', 'cn', 're100']].mean()
    print(f"\nCluster {i} (n={n}):")
    print(profile.round(3).to_dict())


In [ ]:
print("\nANALYSIS 8: UPGRADE/DOWNGRADE RATES")


df['nt_upgrade'] = ((df['2024_NT_Status']=='C') & (df['2025_NT_Status']=='T')).astype(int)
df['nt_downgrade'] = ((df['2024_NT_Status']=='T') & (df['2025_NT_Status']=='C')).astype(int)
df['nz_upgrade'] = ((df['2024_NZ_Status']=='C') & (df['2025_NZ_Status']=='T')).astype(int)
df['nz_downgrade'] = ((df['2024_NZ_Status']=='T') & (df['2025_NZ_Status']=='C')).astype(int)

print(f"NT upgrades: {df['nt_upgrade'].sum()}")
print(f"NT downgrades: {df['nt_downgrade'].sum()}")
print(f"NZ upgrades: {df['nz_upgrade'].sum()}")
print(f"NZ downgrades: {df['nz_downgrade'].sum()}")

In [ ]:
print("\nANALYSIS 9: LOGISTIC FOR CHANGE - WHO CONVERTS 2024->2025")


converters = df[(df['has_nz_2024']==0) & (df['has_nz_2025']==1)]
non_converters = df[(df['has_nz_2024']==0) & (df['has_nz_2025']==0)]
subset = pd.concat([converters, non_converters])

X_change = subset[['has_nt_2024', 'cc_usage', 'cn', 're100']].fillna(0)
y_change = (subset['has_nz_2025']==1).astype(int)

if len(y_change.unique()) > 1:
    lr_change = LogisticRegression(max_iter=1000)
    lr_change.fit(X_change, y_change)
    
    print("Predictors of NZ adoption (among non-NZ in 2024):")
    for feat, coef in zip(X_change.columns, lr_change.coef_[0]):
        odds_ratio = np.exp(coef)
        print(f"{feat}: OR={odds_ratio:.3f}")


In [6]:
print("\nANALYSIS 10: CRAMERS V CORRELATION MATRIX")


vars_corr = ['has_nt', 'nz', 'cc_yes', 'cn', 're100']
n_vars = len(vars_corr)
corr_matrix = np.zeros((n_vars, n_vars))

for i, v1 in enumerate(vars_corr):
    for j, v2 in enumerate(vars_corr):
        if i == j:
            corr_matrix[i, j] = 1.0
        else:
            ct = pd.crosstab(df[v1], df[v2])
            chi2, p, dof, exp = chi2_contingency(ct)
            n = ct.sum().sum()
            cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
            corr_matrix[i, j] = cramers_v

corr_df = pd.DataFrame(corr_matrix, index=vars_corr, columns=vars_corr)
print(corr_df.round(3))


ANALYSIS 10: CRAMERS V CORRELATION MATRIX
        has_nt     nz  cc_yes     cn  re100
has_nt   1.000  0.309   0.099  0.044  0.280
nz       0.309  1.000   0.435  0.472  0.284
cc_yes   0.099  0.435   1.000  0.040  0.150
cn       0.044  0.472   0.040  1.000  0.065
re100    0.280  0.284   0.150  0.065  1.000


In [ ]:
# VISUALIZATIONS
fig1 = go.Figure(data=go.Heatmap(
    z=corr_matrix,
    x=vars_corr,
    y=vars_corr,
    colorscale='Blues'
))
fig1.update_layout(title="Cramers V Correlation Matrix", height=600, width=700)

fig2 = go.Figure(data=[
    go.Bar(name='2024', x=['NT', 'NZ'], y=[df['has_nt_2024'].sum(), df['has_nz_2024'].sum()]),
    go.Bar(name='2025', x=['NT', 'NZ'], y=[df['has_nt_2025'].sum(), df['has_nz_2025'].sum()])
])
fig2.update_layout(title="Commitment Counts 2024 vs 2025", barmode='group')

fig3 = make_subplots(rows=1, cols=2, subplot_titles=['By Sector', 'By Region'])
sector_agg = df.groupby('sector')['has_nz_2025'].mean().sort_values()
region_agg = df.groupby('region')['has_nz_2025'].mean().sort_values()
fig3.add_trace(go.Bar(x=sector_agg.values, y=sector_agg.index, orientation='h'), row=1, col=1)
fig3.add_trace(go.Bar(x=region_agg.values, y=region_agg.index, orientation='h'), row=1, col=2)
fig3.update_layout(title="NZ Adoption Rate by Sector and Region", height=400, width=1000)

import os
os.chdir('/mnt/user-data/outputs')

fig1.write_html('cramers_v_matrix.html')
fig2.write_html('yoy_comparison.html')
fig3.write_html('sector_region_benchmark.html')

chi_df.to_csv('chi_square_tests.csv', index=False)
corr_df.to_csv('correlation_matrix.csv')
trans_ct.to_csv('state_transitions.csv')
df.to_csv('analysis_dataset.csv', index=False)

print("\nFiles saved")

In [ ]:
import pandas as pd

# Check available sheets
xl = pd.ExcelFile(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\historic new .xlsx')
print("Available sheets:", xl.sheet_names)

# Load each year
for year in ['2021', '2022', '2023', '2024', '2025']:
    if year in xl.sheet_names:
        df_year = pd.read_excel(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\historic new .xlsx', sheet_name=year, header=1, nrows=5)
        print(f"\n{year} - Columns:", df_year.columns.tolist()[:10])

Available sheets: ['CN_NZ', '2025', '2024', '2023', '2022', '2021', '2020', '2019', 'sbti', 'sbti evolution ', 'ref']

2021 - Columns: ['3M', 382, 'United States', 'Chemicals', 'Y', 2050, 2019, 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9']

2022 - Columns: ['Walmart', 1, 'United States of America (USA)', 'Retail', 'Y', 2025, 'Y.1', 'T', '2025.1', 'Unnamed: 9']

2023 - Columns: [1, 'Walmart', 'United States of America (USA)', 'Retailing', '$611,289', '$11,680', 'Y', 2035, 2015, 'T']

2024 - Columns: [489, '3M', 'Chemicals', 'United States of America (USA)', 'North America', 'Y', 2019, 2050, 'N', 'Committed']

2025 - Columns: ['\xa0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', '\xa0.1', 'Unnamed: 7']


In [3]:
import pandas as pd

xl = pd.ExcelFile(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\historic new .xlsx')

for year in ['2021', '2022', '2023', '2024', '2025']:
    print(f"\n{'='*60}")
    print(f"{year}")
    print('='*60)
    df_year = pd.read_excel(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\historic new .xlsx', sheet_name=year, nrows=3)
    print(df_year.iloc[:3, :12])


2021
               company  Rank   Headquarters                           Sector  \
0                   3M   382  United States                        Chemicals   
1                  ABB   410    Switzerland                 Industrial goods   
2  Abbott Laboratories   350  United States  Health care and pharmaceuticals   

  RE100 goal    From  Announced Target (T) or committed (C)   From.1  \
0          Y  2050.0     2019.0                          NaN     NaN   
1          Y  2030.0     2021.0                            T  2030.0   
2        NaN     NaN        NaN                          NaN     NaN   

   Announced.1  \
0          NaN   
1       2021.0   
2          NaN   

  Carbon Neutral:\nCompany (C), product (P), unspecified (U) or future  \
0                                                  C                     
1                                                  C                     
2                                                NaN                     

  Scope of com

In [ ]:


required_columns = {
    'company': 'company name',
    'has_nt': 'binary 0/1 for SBTi near-term target',
    'nz': 'binary 0/1 for net zero commitment',
    'cn': 'binary 0/1 for carbon neutral',
    'cc_yes': 'binary 0/1 for carbon credit usage',
    're100': 'binary 0/1 for RE100'
}

# For each year (2021-2025), create a dataframe with these columns:
df_2021 = pd.DataFrame({
    'company': [...],
    'has_nt': [...],
    'nz': [...],
    'cn': [...],
    'cc_yes': [...],
    're100': [...]
})

df_2022 = pd.DataFrame({
    'company': [...],
    'has_nt': [...],
    'nz': [...],
    'cn': [...],
    'cc_yes': [...],
    're100': [...]
})

df_2023 = pd.DataFrame({
    'company': [...],
    'has_nt': [...],
    'nz': [...],
    'cn': [...],
    'cc_yes': [...],
    're100': [...]
})

df_2024 = pd.DataFrame({
    'company': [...],
    'has_nt': [...],
    'nz': [...],
    'cn': [...],
    'cc_yes': [...],
    're100': [...]
})

df_2025 = pd.DataFrame({
    'company': ['company'],
    'has_nt': ['All SBTi ST'],
    'nz': ['NZ'],
    'cn': ['CN'],
    'cc_yes': ['Carbon Credits (Y)'],
    're100': ['All RE']
})

# Then merge all years:
df_all = df_2021.assign(year=2021).append([
    df_2022.assign(year=2022),
    df_2023.assign(year=2023),
    df_2024.assign(year=2024),
    df_2025.assign(year=2025)
])

In [3]:
import pandas as pd
import numpy as np

# ASSUMING df_all EXISTS with columns: company, year, has_nt, nz, cn, cc_yes, re100

print(f"Combined data: {df_all.shape}")
print(df_all.groupby('year').size())

# ============================================================
# ANALYSIS 1: CN → NZ PROGRESSION
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 1: CN → NZ PROGRESSION")
print("="*60)

cn_2021 = set(df_all[(df_all['year']==2021) & (df_all['cn']==1)]['company'])
print(f"Companies with CN in 2021: {len(cn_2021)}")

for year in [2021, 2022, 2023, 2024, 2025]:
    df_year = df_all[df_all['year']==year]
    cohort = df_year[df_year['company'].isin(cn_2021)]
    nz_rate = cohort['nz'].mean()
    cn_rate = cohort['cn'].mean()
    print(f"{year}: {nz_rate:.1%} have NZ, {cn_rate:.1%} still CN (n={len(cohort)})")

df_2021 = df_all[df_all['year']==2021].set_index('company')
df_2025 = df_all[df_all['year']==2025].set_index('company')
cn_to_nz = [co for co in cn_2021 if co in df_2021.index and co in df_2025.index 
            and df_2021.loc[co, 'cn']==1 and df_2025.loc[co, 'nz']==1]
print(f"Converted CN→NZ: {len(cn_to_nz)} ({len(cn_to_nz)/len(cn_2021):.1%})")

# ============================================================
# ANALYSIS 2: NT → NZ CONVERSION
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 2: NT → NZ CONVERSION")
print("="*60)

nt_2021 = set(df_all[(df_all['year']==2021) & (df_all['has_nt']==1)]['company'])
print(f"Companies with NT in 2021: {len(nt_2021)}")

for year in [2021, 2022, 2023, 2024, 2025]:
    cohort = df_all[(df_all['year']==year) & (df_all['company'].isin(nt_2021))]
    nz_rate = cohort['nz'].mean()
    print(f"{year}: {nz_rate:.1%} have NZ (n={len(cohort)})")

# ============================================================
# ANALYSIS 3A: NZ → CC
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 3A: NZ → CC (Does NZ drive credit adoption?)")
print("="*60)

nz_2021_no_cc = set(df_all[(df_all['year']==2021) & (df_all['nz']==1) & (df_all['cc_yes']==0)]['company'])
print(f"Companies with NZ but no CC in 2021: {len(nz_2021_no_cc)}")

for year in [2022, 2023, 2024, 2025]:
    cohort = df_all[(df_all['year']==year) & (df_all['company'].isin(nz_2021_no_cc))]
    cc_rate = cohort['cc_yes'].mean()
    print(f"{year}: {cc_rate:.1%} now use CC (n={len(cohort)})")

# ============================================================
# ANALYSIS 3B: CC → NZ
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 3B: CC → NZ (Do credits enable NZ?)")
print("="*60)

cc_2021_no_nz = set(df_all[(df_all['year']==2021) & (df_all['cc_yes']==1) & (df_all['nz']==0)]['company'])
print(f"Companies with CC but no NZ in 2021: {len(cc_2021_no_nz)}")

for year in [2022, 2023, 2024, 2025]:
    cohort = df_all[(df_all['year']==year) & (df_all['company'].isin(cc_2021_no_nz))]
    nz_rate = cohort['nz'].mean()
    print(f"{year}: {nz_rate:.1%} now have NZ (n={len(cohort)})")

# ============================================================
# ANALYSIS 4: COMMITMENT PATHWAYS
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 4: COMMITMENT PATHWAYS")
print("="*60)

def get_state(row):
    if row['nz']==1 and row['has_nt']==1 and row['cc_yes']==1:
        return 'NZ+NT+CC'
    elif row['nz']==1 and row['has_nt']==1:
        return 'NZ+NT'
    elif row['nz']==1 and row['cc_yes']==1:
        return 'NZ+CC'
    elif row['nz']==1:
        return 'NZ only'
    elif row['cn']==1:
        return 'CN'
    elif row['has_nt']==1:
        return 'NT only'
    else:
        return 'None'

df_all['state'] = df_all.apply(get_state, axis=1)

transitions = []
for co in df_all['company'].unique():
    co_data = df_all[df_all['company']==co].sort_values('year')
    if len(co_data) >= 2:
        for i in range(len(co_data)-1):
            from_state = co_data.iloc[i]['state']
            to_state = co_data.iloc[i+1]['state']
            if from_state != to_state:
                transitions.append({'from': from_state, 'to': to_state})

df_trans = pd.DataFrame(transitions)
print(f"Total transitions: {len(df_trans)}")
print("\nTop pathways:")
print(df_trans.groupby(['from', 'to']).size().sort_values(ascending=False).head(10))

# ============================================================
# ANALYSIS 5: DROPOUT RATES
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 5: COMMITMENT STABILITY (2021→2025)")
print("="*60)

for commitment in ['has_nt', 'nz', 'cn', 'cc_yes', 're100']:
    had_2021 = set(df_all[(df_all['year']==2021) & (df_all[commitment]==1)]['company'])
    if len(had_2021) > 0:
        still_2025 = df_all[(df_all['year']==2025) & (df_all['company'].isin(had_2021))][commitment].sum()
        retention = still_2025 / len(had_2021)
        print(f"{commitment}: {retention:.1%} retention ({still_2025}/{len(had_2021)})")

# ============================================================
# ANALYSIS 6: GROWTH RATES
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 6: GROWTH RATES 2021-2025")
print("="*60)

growth = []
for year in [2021, 2022, 2023, 2024, 2025]:
    df_year = df_all[df_all['year']==year]
    growth.append({
        'year': year,
        'nt': df_year['has_nt'].sum(),
        'nz': df_year['nz'].sum(),
        'cn': df_year['cn'].sum(),
        'cc': df_year['cc_yes'].sum()
    })

df_growth = pd.DataFrame(growth)
print(df_growth)

for col in ['nt', 'nz', 'cn', 'cc']:
    start = df_growth[df_growth['year']==2021][col].values[0]
    end = df_growth[df_growth['year']==2025][col].values[0]
    if start > 0:
        growth_rate = (end - start) / start
        print(f"{col}: {growth_rate:+.1%} growth")

# ============================================================
# ANALYSIS 7: COHORT ANALYSIS
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 7: NZ ADOPTION COHORTS - CC USAGE IN 2025")
print("="*60)

nz_adopters = {}
for co in df_all['company'].unique():
    nz_years = df_all[(df_all['company']==co) & (df_all['nz']==1)]['year'].values
    if len(nz_years) > 0:
        nz_adopters[co] = nz_years[0]

for cohort_year in [2021, 2022, 2023, 2024, 2025]:
    cohort_cos = [co for co, yr in nz_adopters.items() if yr == cohort_year]
    if len(cohort_cos) > 0:
        cohort_2025 = df_all[(df_all['year']==2025) & (df_all['company'].isin(cohort_cos))]
        cc_rate = cohort_2025['cc_yes'].mean()
        print(f"{cohort_year} NZ adopters: {cc_rate:.1%} use CC in 2025 (n={len(cohort_cos)})")

# ============================================================
# ANALYSIS 8: TIME TO NZ
# ============================================================
print("\n" + "="*60)
print("ANALYSIS 8: TIME TO NZ ADOPTION")
print("="*60)

time_to_nz = []
for co in df_all['company'].unique():
    co_data = df_all[df_all['company']==co].sort_values('year')
    first_action = co_data[(co_data[['has_nt', 'cn', 'cc_yes', 're100']].sum(axis=1) > 0)]
    if len(first_action) == 0:
        continue
    
    start_year = first_action.iloc[0]['year']
    had_cc_start = first_action.iloc[0]['cc_yes']
    nz_year = co_data[co_data['nz']==1]['year'].values
    
    if len(nz_year) > 0:
        time_to_nz.append({
            'time': nz_year[0] - start_year,
            'had_cc_start': had_cc_start
        })

df_survival = pd.DataFrame(time_to_nz)
print(f"Companies that adopted NZ: {len(df_survival)}")
print(f"Average time to NZ: {df_survival['time'].mean():.1f} years")
print(f"With CC from start: {df_survival[df_survival['had_cc_start']==1]['time'].mean():.1f} years")
print(f"Without CC: {df_survival[df_survival['had_cc_start']==0]['time'].mean():.1f} years")

NameError: name 'df_all' is not defined

In [19]:
import pandas as pd
import openpyxl

file_path = r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\historic new.xlsx'

name_mappings = {
    'AmerisourceBergen': 'Cencora',
    'Amer International Group': 'Cencora',
    'Electricité de France': 'Electricite de France',
    'Deutsche Post DHL Group': 'DHL Group',
    'Nippon Telegraph and Telephone': 'NTT',
    'Brookfield Asset Management': 'Brookfield',
    'SK Group': 'SK',
    'General Electric': 'General Electric (GE Aerospace)',
    'Bunge': 'Bunge Global',
    'POSCO': 'POSCO Holdings',
    'PKN ORLEN Group': 'Orlen',
    'América Móvil': 'America Movil',
    'Raízen': 'Raizen',
    'Mitsubishi Corp': 'Mitsubishi',
    'International Business Machines': 'IBM',
    'Raytheon Technologies': 'RTX',
    'World Fuel Services': 'World Kinect',
    'Anthem': 'Elevance Health',
    'Daimler': 'Mercedes-Benz Group',
    'Facebook': 'Meta Platforms',
    'Royal Dutch Shell': 'Shell',
    'Sinochem': 'Sinochem Holdings',
    'Panasonic': 'Panasonic Holdings',
    'GlaxoSmithKline': 'GSK',
    'ViacomCBS': 'Paramount Global',
    'Synnex': 'TD Synnex',
}

wb = openpyxl.load_workbook(file_path)

for sheet in wb.worksheets:
    for row in sheet.iter_rows():
        for cell in row:
            if cell.value in name_mappings:
                cell.value = name_mappings[cell.value]

wb.save(file_path)

In [23]:
import pandas as pd

file_path = r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\historic new.xlsx'

# Load portfolio sheet to see headers
portfolio = pd.read_excel(file_path, sheet_name='target portfolio ', header=0)
print(portfolio.columns.tolist())

['company ', 'country', 'sector ', '2021 cc yes/no', '2022 cc yes/no', '2023 cc yes/no', '2024 cc yes/no', '2025 cc yes/no', '2021 cn ', '2022 cn ', '2023 cn ', '2024 cn ', '2025 cn ', '2021 re', '2022 re', '2023 re', '2024 re', '2025 re', '2021 sbti nt', '2022 sbti nt', '2023 sbti nt', '2024 sbti nt', '2025 sbti nt', '2021 nz', '2022 nz', '2023 nz', '2024 nz', '2025 nz']


In [15]:
import pandas as pd

file_path = r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\historic new.xlsx'

y2021 = pd.read_excel(file_path, sheet_name='2021', header=1)
y2022 = pd.read_excel(file_path, sheet_name='2022', header=1)
y2023 = pd.read_excel(file_path, sheet_name='2023', header=1)
y2024 = pd.read_excel(file_path, sheet_name='2024', header=1)
y2025 = pd.read_excel(file_path, sheet_name='2025', header=1)

col_map_2021 = {y2021.columns[0]: 'company', 'CN': 'cn', 'NZ': 'nz', 'RE100': 're', 'SBTi': 'sbti_nt'}
col_map_2022 = {y2022.columns[0]: 'company', 'CN': 'cn', 'ZN': 'nz', 'sbt nt ': 'sbti_nt', 'CC': 'cc', 'RE100 goal': 're'}
col_map_2023 = {y2023.columns[1]: 'company', 'CN': 'cn', 'NZ': 'nz', 'SBTi': 'sbti_nt', 'CC': 'cc', 'RE100 goal': 're'}
col_map_2024 = {y2024.columns[1]: 'company', 'CN ': 'cn', 'NZ': 'nz', 'sbti nt': 'sbti_nt', 'Any RE100': 're', 'will use cabon credits': 'cc'}
col_map_2025 = {y2025.columns[0]: 'company', 'CN': 'cn', 'NZ': 'nz', 'All RE': 're', 'All SBTi ST': 'sbti_nt', 'Carbon Credits (Y)': 'cc'}


y2021 = y2021.rename(columns=col_map_2021)[['company', 'cn', 'nz', 're', 'sbti_nt']]
y2022 = y2022.rename(columns=col_map_2022)[['company', 'cn', 'nz', 're', 'sbti_nt', 'cc']]
y2023 = y2023.rename(columns=col_map_2023)[['company', 'cn', 'nz', 're', 'sbti_nt', 'cc']]
y2024 = y2024.rename(columns=col_map_2024)[['company', 'cn', 'nz', 're', 'sbti_nt', 'cc']]
y2025 = y2025.rename(columns=col_map_2025)[['company', 'cn', 'nz', 're', 'sbti_nt', 'cc']]

all_companies = set()
for df in [y2021, y2022, y2023, y2024, y2025]:
    all_companies.update(df['company'].dropna())

portfolio = pd.DataFrame({'company ': sorted(all_companies)})

for year, df in [('2021', y2021), ('2022', y2022), ('2023', y2023), ('2024', y2024), ('2025', y2025)]:
    df_year = df.rename(columns={
        'cn': f'{year} cn ',
        'nz': f'{year} nz',
        're': f'{year} re',
        'sbti_nt': f'{year} sbti nt',
        'cc': f'{year} cc yes/no'
    })
    portfolio = portfolio.merge(df_year, left_on='company ', right_on='company', how='left').drop(columns=['company'])

with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    portfolio.to_excel(writer, sheet_name='target portfolio', startrow=1, index=False, header=False)

In [17]:
print(portfolio.head(20))
print(f"\nShape: {portfolio.shape}")
print(f"\nNull counts:\n{portfolio.isnull().sum()}")

                      company   2021 cn   2021 nz  2021 re  2021 sbti nt  \
0                           3M       NaN      NaN      NaN           NaN   
1                          ABB       NaN      NaN      NaN           NaN   
2                          ACS       NaN      NaN      NaN           NaN   
3                         AEON       NaN      NaN      NaN           NaN   
4                    AIA Group       NaN      NaN      NaN           NaN   
5                          AIG       NaN      NaN      NaN           NaN   
6           ANZ Group Holdings       NaN      NaN      NaN           NaN   
7                         AT&T       NaN      NaN      NaN           NaN   
8                          AXA       NaN      NaN      NaN           NaN   
9                       AbbVie       NaN      NaN      NaN           NaN   
10         Abbott Laboratories       NaN      NaN      NaN           NaN   
11                   Accenture       NaN      NaN      NaN           NaN   
12          

In [1]:
import pandas as pd
from openpyxl import load_workbook

file_path = r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\historic new.xlsx'

wb = load_workbook(file_path)
ws = wb['target portfolio']

for r_idx, row in enumerate(portfolio.itertuples(index=False), start=2):
    for c_idx, value in enumerate(row, start=1):
        ws.cell(row=r_idx, column=c_idx, value=value)

wb.save(file_path)

NameError: name 'portfolio' is not defined

In [18]:
import pandas as pd

file_path = r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\historic new.xlsx'

y2021 = pd.read_excel(file_path, sheet_name='2021', header=1)
y2022 = pd.read_excel(file_path, sheet_name='2022', header=1)
y2023 = pd.read_excel(file_path, sheet_name='2023', header=1)
y2024 = pd.read_excel(file_path, sheet_name='2024', header=1)
y2025 = pd.read_excel(file_path, sheet_name='2025', header=1)

print("2021 columns with 'CN', 'NZ', 'RE', 'SBT':")
print([col for col in y2021.columns if any(x in str(col).upper() for x in ['CN', 'NZ', 'RE100', 'SBT', 'CARBON', 'NET'])])

print("\n2022 columns with 'CN', 'NZ', 'RE', 'SBT', 'CC':")
print([col for col in y2022.columns if any(x in str(col).upper() for x in ['CN', 'NZ', 'RE100', 'SBT', 'CC', 'CARBON', 'NET'])])

print("\n2023 columns with 'CN', 'NZ', 'RE', 'SBT', 'CC':")
print([col for col in y2023.columns if any(x in str(col).upper() for x in ['CN', 'NZ', 'RE100', 'SBT', 'CC', 'CARBON', 'NET', 'SCIENCE'])])

print("\n2024 columns with 'CN', 'NZ', 'RE', 'SBT', 'CC':")
print([col for col in y2024.columns if any(x in str(col).upper() for x in ['CN', 'NZ', 'RE100', 'SBT', 'CC', 'CARBON', 'NET'])])

print("\n2025 columns with 'CN', 'NZ', 'RE', 'SBT', 'CC':")
print([col for col in y2025.columns if any(x in str(col).upper() for x in ['CN', 'NZ', 'RE100', 'SBT', 'CC', 'CARBON', 'NET'])])

2021 columns with 'CN', 'NZ', 'RE', 'SBT':
['RE100 goal', 'Carbon Neutral:\nCompany (C), product (P), unspecified (U) or future', 'Net Zero: \nCompany (C), product (P), value chain (V) or unspecified (U) (* = no SBT)', 'CN', 'NZ', 'RE100', 'SBTi']

2022 columns with 'CN', 'NZ', 'RE', 'SBT', 'CC':
['RE100 goal', 'SBT (Y = yes)', 'SBTi Statud', 'Net zero year (SBTi)', 'End target year (SBTi and others where no SBT or SBT "C")', "Scope of target\nV = value chain, C = company, S = subsidiary, U = unclear, x = less than 25% rev, + = the scope of the later target, SBTi = target approved under SBTi's Net Zero standard, SBTi? = have a net zero target listed by SBTi but unclear whether this is under the new standard", 'CN', 'sbt nt ', 'CC']

2023 columns with 'CN', 'NZ', 'RE', 'SBT', 'CC':
['RE100 goal', 'Net zero committed (SBTi) (Y = yes, N = no)', 'SBTi NZ approved', 'Net zero year (SBTi)', 'CN', 'NZ', 'SBTi', 'CC', 'NZ year', 'SBTi short term', 'RE100', 'Carbon Neutral', 'Net Zero', 'CC ove

In [35]:
import pandas as pd

file_path = r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\historic new.xlsx'

y2021 = pd.read_excel(file_path, sheet_name='2021', header=1)
y2022 = pd.read_excel(file_path, sheet_name='2022', header=1)
y2023 = pd.read_excel(file_path, sheet_name='2023', header=1)
y2024 = pd.read_excel(file_path, sheet_name='2024', header=1)
y2025 = pd.read_excel(file_path, sheet_name='2025', header=1)

col_map_2021 = {y2021.columns[0]: 'company', 'CN': 'cn', 'NZ': 'nz', 'RE100': 're', 'SBTi': 'sbti_nt'}
col_map_2022 = {y2022.columns[0]: 'company', 'CN': 'cn', 'sbt nt ': 'sbti_nt', 'CC': 'cc', 'RE100 goal': 're'}
col_map_2023 = {y2023.columns[0]: 'company', 'CN': 'cn', 'NZ': 'nz', 'SBTi': 'sbti_nt', 'CC': 'cc', 'RE100': 're'}
col_map_2024 = {'Company Name': 'company', 'CN ': 'cn', 'NZ': 'nz', 'sbti nt': 'sbti_nt', 'Any RE100': 're', 'will use cabon credits': 'cc'}
col_map_2025 = {y2025.columns[0]: 'company', 'CN': 'cn', 'NZ': 'nz', 'All RE': 're', 'All SBTi ST': 'sbti_nt', 'Carbon Credits (Y)': 'cc'}

y2021 = y2021.rename(columns=col_map_2021)[['company', 'cn', 'nz', 're', 'sbti_nt']]
y2022 = y2022.rename(columns=col_map_2022)[['company', 'cn', 're', 'sbti_nt', 'cc']]
y2023 = y2023.rename(columns=col_map_2023)[['company', 'cn', 'nz', 're', 'sbti_nt', 'cc']]
y2024 = y2024.rename(columns=col_map_2024)[['company', 'cn', 'nz', 're', 'sbti_nt', 'cc']]
y2025 = y2025.rename(columns=col_map_2025)[['company', 'cn', 'nz', 're', 'sbti_nt', 'cc']]

y2021 = y2021[y2021['company'].apply(lambda x: isinstance(x, str))]
y2022 = y2022[y2022['company'].apply(lambda x: isinstance(x, str))]
y2023 = y2023[y2023['company'].apply(lambda x: isinstance(x, str))]
y2024 = y2024[y2024['company'].apply(lambda x: isinstance(x, str))]
y2025 = y2025[y2025['company'].apply(lambda x: isinstance(x, str))]

all_companies = set()
for df in [y2021, y2022, y2023, y2024, y2025]:
    all_companies.update(df['company'])

portfolio = pd.DataFrame({'company ': sorted(all_companies)})

for year, df in [('2021', y2021), ('2022', y2022), ('2023', y2023), ('2024', y2024), ('2025', y2025)]:
    rename_dict = {'cn': f'{year} cn ', 'nz': f'{year} nz', 're': f'{year} re', 'sbti_nt': f'{year} sbti nt'}
    if 'cc' in df.columns:
        rename_dict['cc'] = f'{year} cc yes/no'
    df_year = df.rename(columns=rename_dict)
    portfolio = portfolio.merge(df_year, left_on='company ', right_on='company', how='left').drop(columns=['company'])

with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    portfolio.to_excel(writer, sheet_name='target portfolio', startrow=1, index=False)
    

In [34]:
print(portfolio.head(20))
print(f"\nShape: {portfolio.shape}")
print(f"\nNull counts:\n{portfolio.isnull().sum()}")



all_companies

                      company   2021 cn   2021 nz  2021 re  2021 sbti nt  \
0                           3M       1.0      NaN      1.0           NaN   
1                          ABB       1.0      NaN      1.0           1.0   
2                          ACS       NaN      NaN      NaN           NaN   
3                         AEON       NaN      1.0      1.0           1.0   
4                    AIA Group       NaN      NaN      NaN           NaN   
5                          AIG       NaN      NaN      NaN           NaN   
6           ANZ Group Holdings       NaN      NaN      NaN           NaN   
7                         AT&T       1.0      NaN      NaN           1.0   
8                          AXA       1.0      1.0      1.0           NaN   
9                       AbbVie       NaN      NaN      NaN           NaN   
10         Abbott Laboratories       NaN      NaN      NaN           NaN   
11                   Accenture       NaN      1.0      1.0           1.0   
12          

{'3M',
 'ABB',
 'ACS',
 'AEON',
 'AIA Group',
 'AIG',
 'ANZ Group Holdings',
 'AT&T',
 'AXA',
 'AbbVie',
 'Abbott Laboratories',
 'Accenture',
 'Achmea',
 'Aegon',
 'Agricultural Bank of China',
 'Air France-KLM Group',
 'Airbus',
 'Aisin',
 'Albertsons',
 'Alfresa Holdings',
 'Alibaba Group Holding',
 'Alimentation Couche-Tard',
 'Allianz',
 'Allstate',
 'Alphabet',
 'Aluminum Corp. of China',
 'Amazon',
 'Amer International Group',
 'America Movil',
 'American Airlines Group',
 'American Express',
 'American International Group',
 'AmerisourceBergen',
 'Amgen',
 'América Móvil',
 'Anglo American',
 'Anheuser-Busch InBev',
 'Anhui Conch Group',
 'Ansteel Group',
 'Anthem',
 'Apollo Global Management',
 'Apple',
 'ArcelorMittal',
 'Archer Daniels Midland',
 'Arrow Electronics',
 'Assicurazioni Generali',
 'AstraZeneca',
 'Aviation Industry Corp. of China',
 'Aviva',
 'BAE Systems',
 'BASF',
 'BHP Group',
 'BMW Group',
 'BNP Paribas',
 'BP',
 'BT Group',
 'BYD',
 'Banco Bilbao Vizcaya A